# Redshift e Classificazione delle Stelle

Il **redshift** (spostamento verso il rosso) è un fenomeno fondamentale in cosmologia e astrofisica. Si verifica quando la luce emessa da un oggetto celeste viene spostata verso lunghezze d'onda più lunghe (il rosso dello spettro) a causa dell'espansione dell'Universo.

### Formula del redshift

$$z = \frac{\lambda_{\text{osservata}} - \lambda_{\text{emessa}}}{\lambda_{\text{emessa}}}$$

Dove:
- $z$ è il redshift (adimensionale)
- $\lambda_{\text{osservata}}$ è la lunghezza d'onda misurata
- $\lambda_{\text{emessa}}$ è la lunghezza d'onda a riposo

Per $z > 0$ l'oggetto si allontana (Universo in espansione), per $z < 0$ si avvicina (spostamento verso il blu).

### Obiettivi di questo notebook

1. Configurare l'ambiente per lavorare con i database DESI
2. Scaricare e installare i pacchetti `specprod-db`, `desiutil`, `desitarget`, `desisurvey`
3. Connettersi a un database PostgreSQL contenente dati spettroscopici
4. Estrarre e analizzare dati di redshift per classificazione stellare

## 1. Configurazione dell'ambiente

Prima di lavorare con il database DESI, eseguire le seguenti operazioni:

1. Installare Jupyter Notebook per lavorare con file locali
2. Configurare un environment Python (simile a `conda env`) con le librerie necessarie
3. Installare **PostgreSQL** e avviare il servizio:
   - Premi `Win + R`, digita `services.msc` e premi Invio
   - Trova il servizio PostgreSQL (es. `postgresql-x64-15`)
   - Verifica che sia in esecuzione; altrimenti, avvialo
4. Installare `sqlalchemy` e `psycopg2` per connettersi a PostgreSQL da Python

## 2. Verifica dell'ambiente Python

In [ ]:
import sys
import os

print("Eseguibile Python:", sys.executable)
print("Directory di lavoro:", os.getcwd())

## 3. Download dei pacchetti DESI

Cloniamo i repository GitHub necessari per accedere ai dati DESI (Dark Energy Spectroscopic Instrument).

### Come installare correttamente un progetto Git su Jupyter:
1. Clona il repository
2. Entra nella cartella del progetto
3. Esegui `python setup.py install`
4. Ripeti per ogni repository

In [ ]:
!git clone https://github.com/desihub/specprod-db.git

In [ ]:
!git clone https://github.com/desihub/desiutil.git

In [ ]:
!git clone https://github.com/desihub/desitarget.git

In [ ]:
!git clone https://github.com/desihub/desisurvey.git

## 4. Import delle librerie necessarie

Nota: alcuni moduli potrebbero non essere disponibili o richiedere configurazioni aggiuntive. I commenti indicano import opzionali o problematici.

In [ ]:
# Librerie standard
import os
import sys
import itertools
from types import MethodType

# Librerie scientifiche
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import fontManager, FontProperties

# Connessione database
from sqlalchemy import __version__ as sqlalchemy_version
from sqlalchemy import inspect, and_
from sqlalchemy.sql import func
from sqlalchemy.orm import aliased

# Astropy
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import ICRS

# DESI software
from desiutil.log import get_logger, DEBUG
from desitarget.targetmask import desi_mask, mws_mask, bgs_mask
# from desisim.spec_qa import redshifts as dsq_z
from desisurvey import __version__ as desisurvey_version
# from desisurvey.ephem import get_ephem, get_object_interpolator
# from desisurvey.utils import get_observer
# from specprodDB import __version__ as specprodDB_version
# import specprodDB.load as db

## 5. Configurazione della produzione spettroscopica

Impostiamo la variabile d'ambiente `SPECPROD` per selezionare il dataset DESI (es. `fuji` o `guadalupe`).

In [ ]:
# Imposta la produzione spettroscopica
specprod = os.environ['SPECPROD'] = 'fuji'  # o 'guadalupe' se necessario

# Directory di output per effemeridi
os.environ['DESISURVEY_OUTPUT'] = '/global/cfs/cdirs/desi/public/epo/example_files'

# Directory di lavoro
workingdir = os.getcwd()

print(f"SPECPROD = {specprod}")
print(f"Working directory = {workingdir}")

## 6. Verifica dell'import di specprodDB

Controlliamo se il modulo `specprodDB` è installato correttamente.

In [ ]:
try:
    import specprodDB
    print("specprodDB installato in:", specprodDB.__file__)
except ImportError as e:
    print("specprodDB non trovato. Installalo con 'python setup.py install' nella cartella specprod-db.")
    print(f"Errore: {e}")

## 7. Connessione al database PostgreSQL

Il modulo `specprodDB.load` permette di connettersi al database PostgreSQL remoto di DESI.

**Nota sugli errori comuni:**
- `ImportError: cannot import name 'SafeConfigParser'`: Questo errore si verifica con Python 3.12 perché `SafeConfigParser` è stato rimosso da `configparser`. Soluzione: aggiornare `specprodDB` o usare Python 3.10/3.11.
- **Credenziali di accesso**: Se compare un `RuntimeError`, aggiungere le credenziali al file `$HOME/.pgpass`.

In [ ]:
try:
    import specprodDB.load as db
    db.log = get_logger()
    try:
        postgresql = db.setup_db(
            schema=specprod,
            hostname='specprod-db.desi.lbl.gov',
            username='desi_public'
        )
        print("Connessione al database riuscita!")
    except RuntimeError:
        print("\nÈ necessario aggiungere le credenziali di accesso al file $HOME/.pgpass.")
        print("Esegui il comando:")
        print("cat /global/common/software/desi/desi_public.pgpass >> ~/.pgpass; chmod 600 ~/.pgpass")
except ImportError as e:
    print(f"Errore di import: {e}")
    print("\nNota: specprodDB.load usa 'SafeConfigParser' che non esiste più in Python 3.12.")
    print("Soluzioni possibili:")
    print("  - Usare Python 3.10 o 3.11")
    print("  - Installare una versione aggiornata di specprodDB")
    print("  - Applicare una patch per sostituire SafeConfigParser con ConfigParser")